In [1]:
#!! dit script werkt alleen op een versie van machinaal_test waarin datums en beroepen al werden opgeschoond

In [2]:
#req
import pandas as pd
import re
import numpy as np

In [3]:
#leeftijd opschonen
def clean_data(df):
    df_clean = df.copy()
    
    df_clean['E_YEAR'] = df_clean['E_DATE_clean'].apply(
        lambda x: int(re.search(r'(\d{4})', x).group(1)) if isinstance(x, str) and re.search(r'(\d{4})', x) else np.nan
    )
    
    df_clean['P_AGE_clean'] = df_clean['P_AGE'].apply(clean_p_age)
    
    df_clean['P_BIRTHYEAR'] = df_clean.apply(
        lambda row: extract_birthyear(row['P_BIRTHDATE'], row['E_YEAR']), 
        axis=1
    )
    
    df_clean['P_BIRTHYEAR'] = df_clean['P_BIRTHYEAR'].apply(lambda x: int(x) if pd.notna(x) else np.nan)
    
    df_clean['P_AGE_clean'] = df_clean.apply(
        lambda row: int(row['E_YEAR']) - int(row['P_BIRTHYEAR']) 
            if pd.isna(row['P_AGE_clean']) and pd.notna(row['P_BIRTHYEAR']) and pd.notna(row['E_YEAR']) 
            else row['P_AGE_clean'], 
        axis=1
    )
    
    df_clean = validate_birthyears_ages(df_clean)
    
    return df_clean

def clean_p_age(age_str):
    if pd.isna(age_str) or not isinstance(age_str, str):
        return np.nan
    
    age_str = age_str.lower()
    
    if 'maand' in age_str:
        return 0
    
    if 'kind' in age_str and not re.search(r'\d', age_str):
        return 5
    
    numbers = re.findall(r'\d+', age_str)
    if numbers:
        return int(numbers[0])
    
    return np.nan

def extract_birthyear(birthdate_str, e_year):
    if pd.isna(birthdate_str) or not isinstance(birthdate_str, str):
        return np.nan
    
    birthdate_str = birthdate_str.lower()
    
    # viercijferige jaartallen binnen bereik 1750-1950
    match = re.search(r'\b(17[5-9]\d|18\d{2}|19[0-4]\d|1950)\b', birthdate_str)
    if match:
        return int(match.group(1))
    
    # jaartal aan begin van string
    match = re.match(r'^(18\d{2}|19\d{2})\s', birthdate_str)
    if match:
        return int(match.group(1))
    
    # jaartal aan einde van string, voorafgegaan door maand
    months = (
    r'januari|jan|january|'
    r'februari|feb|february|février|fevrier|'
    r'maart|mrt|march|mar|mars|meert|'
    r'april|apr|avril|'
    r'mei|may|mai|'
    r'juni|jun|june|juin|'
    r'juli|jul|july|juillet|'
    r'augustus|aug|august|oogst|août|aout|'
    r'september|sep|sept|septembre|7ber|7br|'
    r'oktober|okt|oct|october|octobre|8ber|8br|'
    r'november|nov|9ber|9br|'
    r'december|dec|xber|xbr|10ber|10br|décembre|decembre'
)
    match = re.search(fr'({months})\s+(18\d{{2}}|19\d{{2}})\b', birthdate_str)
    if match:
        return int(match.group(2))
    
    # tweecijferige jaartallen
    date_patterns = [
        r'(\d+)\s*[.]\s*(\d+)\s*[.]\s*(\d{2})\s*$',  # vb. 8 . 2 . 08
        r'(\d+)\s*[-./]\s*(\d+)\s*[-./]\s*(\d{2})$',  # vb. 13 - 12 - 63
        r'(\d+)\s*[.]\s*(\d{2})\s*$'  # vb. 10 . 36
    ]
    
    for pattern in date_patterns:
        match = re.search(pattern, birthdate_str)
        if match:
            year_2d = int(match.groups()[-1])
            
            prefix = '18'
            
            if e_year is not None and not pd.isna(e_year):
                if e_year > 1900 and (e_year - int(f'18{year_2d}')) > 100:
                    prefix = '19'
                    
            return int(f'{prefix}{year_2d}')
    
    return np.nan

def validate_birthyears_ages(df):
    df_validated = df.copy()
    
    mask_invalid_birthyear = (
        pd.notna(df_validated['P_BIRTHYEAR']) & 
        ((df_validated['P_BIRTHYEAR'] < 1750) | (df_validated['P_BIRTHYEAR'] > 1950))
    )
    df_validated.loc[mask_invalid_birthyear, 'P_BIRTHYEAR'] = np.nan
    
    mask_invalid_age = (
        pd.notna(df_validated['P_AGE_clean']) & 
        ((df_validated['P_AGE_clean'] < 0) | (df_validated['P_AGE_clean'] > 100))
    )
    df_validated.loc[mask_invalid_age, 'P_AGE_clean'] = np.nan
    
    mask_hisco_age = (
        pd.notna(df_validated['P_PROF_hisco']) & 
        pd.notna(df_validated['P_AGE_clean']) & 
        ((df_validated['P_AGE_clean'] < 5) | (df_validated['P_AGE_clean'] > 80))
    )
    df_validated.loc[mask_hisco_age, 'P_AGE_clean'] = np.nan
    
    invalid_birthyears = mask_invalid_birthyear.sum()
    invalid_ages = mask_invalid_age.sum()
    invalid_hisco_ages = mask_hisco_age.sum()
    
    print(f"Validation complete: {invalid_birthyears} invalid birthyears, {invalid_ages} invalid ages, and {invalid_hisco_ages} invalid ages (>70 with HISCO) set to NaN")
    
    return df_validated

#leeftijdscategorisering
def categorize_age(age):
    if pd.isna(age):
        return np.nan
    
    age = int(age)
    
    if 0 <= age <= 4:
        return 'jong kind'
    elif 5 <= age <= 14:
        return 'kind'
    elif 15 <= age <= 24:
        return 'jongvolwassene'
    elif 25 <= age <= 64:
        return 'volwassene'
    elif age >= 65:
        return 'oudere'
    else:
        return np.nan

if __name__ == "__main__":
    df = pd.read_excel('../../../data/H4/normalisering/machinaal_test_prof.xlsx')
    cleaned_df = clean_data(df)
    
    cleaned_df['P_AGE_categories'] = cleaned_df['P_AGE_clean'].apply(categorize_age)
    
    print(cleaned_df[['P_AGE', 'P_AGE_clean', 'P_AGE_categories']].head(10))
    
    cleaned_df.to_excel('../../../data/H4/normalisering/machinaal_test_age.xlsx', index=False)

Validation complete: 4 invalid birthyears, 29 invalid ages, and 17 invalid ages (>70 with HISCO) set to NaN
     P_AGE  P_AGE_clean P_AGE_categories
0      NaN          NaN              NaN
1      NaN          NaN              NaN
2      NaN          NaN              NaN
3  19 jaar         19.0   jongvolwassene
4      NaN          NaN              NaN
5      NaN          NaN              NaN
6      NaN          NaN              NaN
7      NaN          NaN              NaN
8      NaN          NaN              NaN
9      NaN          NaN              NaN
